# Leakage-audited solar flare forecasting across Solar Cycles 24 and 25

Reproducible pipeline for the paper *Leakage-Audited Solar Flare Forecasting Across Solar Cycles 24 and 25* (Avinash Kumar, ORCID 0009-0008-7613-1185).

**What it does**
1. Downloads public inputs (SWAN-SF addenda flare list, JSOC SHARP keywords, NOAA NCEI GOES flare summary, HEK flare records).
2. Builds hourly 12-hour windows with a 24-hour M+ target for Cycle 24 (2010-2018) and Cycle 25 (2019-2024) with one labelling pipeline.
3. Runs the leakage audit, cross-cycle transfer test, per-year skill and calibration analyses, and makes the figures.

**How to run (Google Colab, CPU is enough):** run the cells in order. Each data step skips work that is already on disk. Total time about 1-1.5 hours, mostly JSOC queries.
If the runtime resets, run the optional *restore* cell (needs a Drive backup made by the *save* cell) or re-run from the top.

**Known notes**
- Cells in section 3 print their results; the figure cell (section 4) uses per-year and calibration numbers copied from the output of the *per-year and calibration* cell. Re-copy them if you change anything upstream.
- Data licences: SWAN-SF addenda CC0; JSOC, NOAA and HEK data are public. Code licence: add one before release.

In [ ]:
!pip -q install drms netCDF4 xarray pyarrow

In [ ]:
import os, re, time, glob, shutil, tarfile, requests, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.chdir("/content"); os.makedirs("swan", exist_ok=True); os.makedirs("pinn_data", exist_ok=True)

# 17 SHARP parameters, 12-hour window, 24-hour horizon
P = ['ABSNJZH','MEANALP','MEANGAM','MEANGBH','MEANGBT','MEANGBZ','MEANJZD','MEANJZH',
     'MEANPOT','MEANSHR','R_VALUE','SAVNCPP','SHRGT45','TOTPOT','TOTUSJH','TOTUSJZ','USFLUX']
W = 12
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36"}
BACKUP = "/content/drive/MyDrive/solar_bench"

def get(url, path):
    if os.path.exists(path): print("have", path); return
    with requests.get(url, headers=UA, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for ch in r.iter_content(1 << 20): f.write(ch)
    print("saved", path, round(os.path.getsize(path)/1e6, 1), "MB", flush=True)

import sklearn, drms, xarray
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| drms", drms.__version__, "| xarray", xarray.__version__)

## 0. Optional: restore a Drive backup (skip on a first run)

In [ ]:
done = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for root, _, fs in os.walk(BACKUP):
        for f in fs:
            rel = os.path.relpath(os.path.join(root, f), BACKUP)
            os.makedirs(os.path.dirname(rel) or ".", exist_ok=True)
            shutil.copy(os.path.join(root, f), rel)
    done = True; print("restored from", BACKUP)
except Exception as e:
    print("restore skipped:", type(e).__name__)

## 1. Download inputs

In [ ]:
import drms, xarray as xr

# 1a) SWAN-SF addenda: integrated GOES flare list with NOAA region numbers (2010-2018)
get("https://dataverse.harvard.edu/api/access/datafile/3642452", "swan/addenda.tar.gz")
if not os.path.exists("swan/addenda/integrated_flare_data"):
    with tarfile.open("swan/addenda.tar.gz") as t: t.extractall("swan/addenda", filter="data")
    print("extracted addenda")

# 1b) SHARP keywords from JSOC at 1-hour cadence
KEYS = ("HARPNUM, T_REC, NOAA_ARS, NOAA_NUM, QUALITY, LAT_FWT, LON_FWT, CRLN_OBS, "
        "USFLUX, MEANGAM, MEANGBT, MEANGBZ, MEANGBH, MEANJZD, TOTUSJZ, MEANALP, MEANJZH, "
        "TOTUSJH, ABSNJZH, SAVNCPP, MEANPOT, TOTPOT, MEANSHR, SHRGT45, R_VALUE, AREA_ACR, SIZE_ACR, NACR")

def jsoc_months(months, out):
    if os.path.exists(out): print("have", out); return
    c = drms.Client(); parts, t0 = [], time.time()
    for m in months:
        nd = m.days_in_month; start = m.strftime("%Y.%m.%d_00:00:00_TAI")
        for a in range(3):
            try: ks = c.query(f"hmi.sharp_cea_720s[][{start}/{nd}d@1h]", key=KEYS); break
            except Exception as e: print("retry", m.date(), type(e).__name__); time.sleep(5)
        else: raise RuntimeError(f"failed {m.date()}")
        ks["year_month"] = m.strftime("%Y-%m"); parts.append(ks)
        if m.month == 12: print(m.year, "done", round(time.time()-t0), "sec", flush=True)
    pd.concat(parts, ignore_index=True).to_csv(out, index=False); print("saved", out)

jsoc_months(pd.date_range("2019-01-01", "2024-12-01", freq="MS"), "pinn_data/sharp_2019_2024_1h.csv.gz")   # Cycle 25
jsoc_months(pd.date_range("2010-05-01", "2018-12-01", freq="MS"), "pinn_data/sharp_2010_2018_1h.csv.gz")   # Cycle 24

# 1c) NOAA NCEI GOES-R XRS flare summary (class + peak time)
nb = "https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes16/l2/data/xrsf-l2-flsum_science/"
get(nb + "sci_xrsf-l2-flsum_g16_s20170209_e20250406_v2-2-1.nc", "flsum_g16.nc")

# 1d) HEK flare records give the NOAA AR number; match to NCEI peak times (+-10 min)
if not os.path.exists("mplus_2019_2025_ar.csv"):
    url = "https://www.lmsal.com/hek/her"
    ret = "event_starttime,event_peaktime,fl_goescls,ar_noaanum,hpc_x,hpc_y,frm_name"
    def hek(t0, t1, frm):
        p = {"cosec": 2, "cmd": "search", "type": "column", "event_type": "fl", "event_starttime": t0, "event_endtime": t1,
             "event_coordsys": "helioprojective", "x1": -1200, "x2": 1200, "y1": -1200, "y2": 1200, "result_limit": 3000,
             "param0": "FRM_Name", "op0": "=", "value0": frm, "return": ret}
        for k in range(3):
            try: return pd.DataFrame(requests.get(url, params=p, headers=UA, timeout=120).json()["result"])
            except Exception: time.sleep(3)
        return pd.DataFrame()
    outl = []
    for frm in ["SWPC", "SSW Latest Events"]:
        for m in pd.date_range("2019-01-01", "2025-04-30", freq="MS"):
            outl.append(hek(m.strftime("%Y-%m-%dT00:00:00"), (m + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%dT23:59:59"), frm))
        print("HEK", frm, "done", flush=True)
    hk = pd.concat(outl, ignore_index=True)
    hk["peak"] = pd.to_datetime(hk.event_peaktime, errors="coerce")
    hk["ar"] = pd.to_numeric(hk.ar_noaanum, errors="coerce").fillna(0)
    hk.loc[(hk.ar > 0) & (hk.ar < 10000), "ar"] += 10000          # 2821 -> 12821
    hk = hk[hk.fl_goescls.astype(str).str.startswith(("M", "X"))].dropna(subset=["peak"])
    d = xr.open_dataset("flsum_g16.nc").to_dataframe().reset_index()
    mx = d[(d.status == "EVENT_PEAK") & (d.time >= "2019-01-01") & d.flare_class.astype(str).str.startswith(("M", "X"))].reset_index(drop=True)
    res = []
    for _, r in mx.iterrows():
        cnd = hk[(hk.peak - r.time).abs() <= pd.Timedelta("10min")]
        res.append((r.time, r.flare_class, len(cnd), cnd.ar.max() if len(cnd) else 0))
    R = pd.DataFrame(res, columns=["peak", "cls", "n_hek", "ar"]); R["year"] = R.peak.dt.year
    R.to_csv("mplus_2019_2025_ar.csv", index=False)
    print("M+ flares:", len(R), "with AR:", (R.ar > 0).sum(), "(expected 1631 / 1527)")
else: print("have mplus_2019_2025_ar.csv")

## 2. Build window tables (same recipe for both cycles)

In [ ]:
# ---- Cycle 25 ----
if not os.path.exists("c25_samples.pkl"):
    c = pd.read_csv("pinn_data/sharp_2019_2024_1h.csv.gz")
    c["time"] = pd.to_datetime(c.T_REC.str.replace("_TAI", "").str.replace(r"^(\d{4})\.(\d\d)\.(\d\d)_", r"\1-\2-\3 ", regex=True))
    c = c[c.QUALITY == 0].sort_values(["HARPNUM", "time"]).reset_index(drop=True)
    c["arset"] = c.NOAA_ARS.astype(str).apply(lambda s: [int(x) for x in re.findall(r"\b\d{5}\b", s)])
    hp_ars = c.groupby("HARPNUM").arset.apply(lambda L: set(a for l in L for a in l))
    keep = hp_ars[hp_ars.apply(len) > 0].index; c = c[c.HARPNUM.isin(keep)]
    Fl = pd.read_csv("mplus_2019_2025_ar.csv", parse_dates=["peak"]); Fl = Fl[Fl.ar > 0]
    span = c.groupby("HARPNUM").time.agg(["min", "max"]); ev = {h: [] for h in keep}
    for _, r in Fl.iterrows():
        for h in keep:
            if int(r.ar) in hp_ars[h] and span.loc[h, "min"] <= r.peak <= span.loc[h, "max"] + pd.Timedelta("24h"):
                ev[h].append(r.peak.to_datetime64())
    rows = []
    for h, d in c.groupby("HARPNUM"):
        d = d.reset_index(drop=True)
        if len(d) < W: continue
        X = d[P].astype("float64"); r = X.rolling(W, min_periods=9)
        f = pd.concat([X.add_suffix("_last"), r.mean().add_suffix("_mean"), r.std().add_suffix("_std"), X.diff(W-1).add_suffix("_delta")], axis=1)
        e = np.sort(np.array(ev[h], dtype="datetime64[ns]")); t = d.time.values
        f["y"] = (np.searchsorted(e, t + np.timedelta64(24, "h"), side="right") > np.searchsorted(e, t, side="right")).astype("int8")
        f["time"] = d.time; f["ar"] = f"c25_{h}"
        rows.append(f.dropna(subset=[x + "_last" for x in P] + [x + "_mean" for x in P]).iloc[W-1:])
    C = pd.concat(rows, ignore_index=True)
    C = C[(C.time >= "2019-02-01") & (C.time <= "2024-12-30")]
    FEc = [x for x in C.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]
    C[FEc] = C[FEc].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    C.to_pickle("c25_samples.pkl")
    print("C25 samples:", C.shape, "pos:", int(C.y.sum()), "(expected 162801 x 71, pos 13289)")
else: print("have c25_samples.pkl")

In [ ]:
# ---- Cycle 24 (same recipe; flares from the SWAN-SF addenda list) ----
if not os.path.exists("c24p_samples.pkl"):
    g = pd.read_csv("swan/addenda/integrated_flare_data/goes_flares_integrated.csv", parse_dates=["peak_time"])
    g = g[g.goes_class.astype(str).str.startswith(("M", "X"))].copy()
    g["ar"] = pd.to_numeric(g.noaa_active_region, errors="coerce").fillna(0).astype(int)
    print("C24 M+ flares:", len(g), "| AR missing:", (g.ar == 0).sum(), "(expected 792 / 33)")
    Fl = g[g.ar > 0]
    c = pd.read_csv("pinn_data/sharp_2010_2018_1h.csv.gz")
    c["time"] = pd.to_datetime(c.T_REC.str.replace("_TAI", "").str.replace(r"^(\d{4})\.(\d\d)\.(\d\d)_", r"\1-\2-\3 ", regex=True))
    c = c[c.QUALITY == 0].sort_values(["HARPNUM", "time"]).reset_index(drop=True)
    c["arset"] = c.NOAA_ARS.astype(str).apply(lambda s: [int(x) for x in re.findall(r"\b\d{5}\b", s)])
    hp_ars = c.groupby("HARPNUM").arset.apply(lambda L: set(a for l in L for a in l))
    keep = hp_ars[hp_ars.apply(len) > 0].index; c = c[c.HARPNUM.isin(keep)]
    span = c.groupby("HARPNUM").time.agg(["min", "max"]); ev = {h: [] for h in keep}
    for _, r in Fl.iterrows():
        for h in keep:
            if r.ar in hp_ars[h] and span.loc[h, "min"] <= r.peak_time <= span.loc[h, "max"] + pd.Timedelta("24h"):
                ev[h].append(r.peak_time.to_datetime64())
    rows = []
    for h, d in c.groupby("HARPNUM"):
        d = d.reset_index(drop=True)
        if len(d) < W: continue
        X = d[P].astype("float64"); r = X.rolling(W, min_periods=9)
        f = pd.concat([X.add_suffix("_last"), r.mean().add_suffix("_mean"), r.std().add_suffix("_std"), X.diff(W-1).add_suffix("_delta")], axis=1)
        e = np.sort(np.array(ev[h], dtype="datetime64[ns]")); t = d.time.values
        f["y"] = (np.searchsorted(e, t + np.timedelta64(24, "h"), side="right") > np.searchsorted(e, t, side="right")).astype("int8")
        f["time"] = d.time; f["ar"] = f"c24_{h}"
        rows.append(f.dropna(subset=[x + "_last" for x in P] + [x + "_mean" for x in P]).iloc[W-1:])
    C24 = pd.concat(rows, ignore_index=True)
    C24 = C24[(C24.time >= "2010-06-01") & (C24.time <= "2018-12-30")]
    FEc = [x for x in C24.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]
    C24[FEc] = C24[FEc].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    C24.to_pickle("c24p_samples.pkl")
    print("C24 samples:", C24.shape, "pos:", int(C24.y.sum()), "(expected 215463 x 71, pos 7995)")
else: print("have c24p_samples.pkl")

### Save checkpoint to Drive (recommended)

In [ ]:
KEEP = ["c24p_samples.pkl", "c25_samples.pkl", "mplus_2019_2025_ar.csv", "flsum_g16.nc",
        "pinn_data/sharp_2019_2024_1h.csv.gz", "pinn_data/sharp_2010_2018_1h.csv.gz"]
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for p in KEEP:
        if os.path.exists(p):
            os.makedirs(os.path.dirname(f"{BACKUP}/{p}"), exist_ok=True); shutil.copy(p, f"{BACKUP}/{p}")
    print("saved to", BACKUP)
except Exception as e: print("save skipped:", type(e).__name__)

## 3. Experiments

### 3.1 Leakage audit with default hyperparameters (Cycle 24, 5-fold; results saved to leakage_audit_c24p.csv)

In [ ]:
import os, shutil, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, GroupKFold
os.chdir("/content")
A = pd.read_pickle("c24p_samples.pkl").reset_index(drop=True)
FE = [x for x in A.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]
X, y = A[FE].values, A.y.values
codes, uars = pd.factorize(A.ar); NA = len(uars)
col = {c: i for i, c in enumerate(FE)}

def mk(name):
    if name in ("1-feat (USFLUX)", "3-feat (USFLUX,TOTUSJH,R_VALUE)"):
        cols = [col["USFLUX_last"]] if name.startswith("1") else [col[c] for c in ["USFLUX_last", "TOTUSJH_last", "R_VALUE_last"]]
        ct = ColumnTransformer([("s", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), cols)])
        return make_pipeline(ct, LogisticRegression(class_weight="balanced", max_iter=300))
    if name == "LR": return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=300))
    if name == "HGB": return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, class_weight="balanced", random_state=0)
    return make_pipeline(StandardScaler(), MLPClassifier((64, 32), early_stopping=True, max_iter=80, random_state=0))

def tss_pt(p, yy, t):
    pr = p >= t; tp=(pr&(yy==1)).sum(); fn=((~pr)&(yy==1)).sum(); fp=(pr&(yy==0)).sum(); tn=((~pr)&(yy==0)).sum()
    return tp/max(tp+fn,1) - fp/max(fp+tn,1)
def best_t(p, yy):
    ts = np.unique(np.quantile(p, np.linspace(.5, .999, 120))); return max(ts, key=lambda t: tss_pt(p, yy, t))
def add_counts(C, idx, p, t):
    pr = p >= t; yy = y[idx]; cd = codes[idx]
    f = lambda w: np.bincount(cd, weights=w.astype(float), minlength=NA)
    C += np.c_[f(pr&(yy==1)), f((~pr)&(yy==1)), f(pr&(yy==0)), f((~pr)&(yy==0))]
def stats(C, nb=200):
    C = C[C.sum(1) > 0]
    def ts(c): s = c.sum(0); return s[0]/max(s[0]+s[1],1) - s[2]/max(s[2]+s[3],1)
    def hs(c):
        tp, fn, fp, tn = c.sum(0); d = (tp+fn)*(fn+tn)+(tp+fp)*(fp+tn); return 2*(tp*tn-fn*fp)/d if d else 0
    idx = np.random.RandomState(1).randint(0, len(C), (nb, len(C))); b = [ts(C[i]) for i in idx]
    return round(ts(C), 3), f"[{np.percentile(b,2.5):.2f},{np.percentile(b,97.5):.2f}]", round(hs(C), 3)

def fit_eval(name, tr, va, te, C):
    m = mk(name).fit(X[tr], y[tr]); t = best_t(m.predict_proba(X[va])[:, 1], y[va])
    add_counts(C, te, m.predict_proba(X[te])[:, 1], t)

rng = np.random.RandomState(0); res = []
splits = {}
splits["random rows (leaky)"] = [(a, te) for a, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y)]
splits["grouped by AR"]       = [(a, te) for a, te in GroupKFold(5).split(X, y, codes)]
tr0 = np.where(A.time < "2014-01-01")[0]; te0 = np.where((A.time >= "2014-01-15") & (~A.ar.isin(set(A.ar.iloc[tr0]))))[0]
splits["forward time + AR-disjoint"] = [(tr0, te0)]

for mn in ["1-feat (USFLUX)", "3-feat (USFLUX,TOTUSJH,R_VALUE)", "LR", "HGB", "MLP"]:
    for sn, folds in splits.items():
        C = np.zeros((NA, 4))
        for a, te in folds:
            if sn.startswith("random"):
                v = rng.rand(len(a)) < .2; tr, va = a[~v], a[v]
            else:
                ua = np.unique(codes[a]); vs = set(rng.choice(ua, int(.2*len(ua)), replace=False)); isv = np.isin(codes[a], list(vs)); tr, va = a[~isv], a[isv]
            fit_eval(mn, tr, va, te, C)
        t, c, h = stats(C); res.append(dict(model=mn, split=sn, TSS=t, CI95=c, HSS2=h))
    print(mn, "done", flush=True)
R = pd.DataFrame(res); print(R.to_string(index=False))
R.to_csv("leakage_audit_c24p.csv", index=False)

### 3.2 Leakage audit with tuned models, 5-fold, paired CI (saves leakage_tuned_5fold.csv)

In [ ]:
import time
from sklearn.model_selection import StratifiedKFold, GroupKFold
A = pd.read_pickle("c24p_samples.pkl").reset_index(drop=True)
FE = [x for x in A.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]
X, y = A[FE].values, A.y.values
codes, uars = pd.factorize(A.ar); NA = len(uars)

TUNED = {  # selected by grouped CV in 3.4
 "LR (tuned)":  lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.003, class_weight="balanced", max_iter=400)),
 "HGB (tuned)": lambda: HistGradientBoostingClassifier(class_weight="balanced", random_state=0, learning_rate=0.06, max_iter=100, max_leaf_nodes=8, min_samples_leaf=200, l2_regularization=1),
 "MLP (tuned)": lambda: make_pipeline(StandardScaler(), MLPClassifier((64, 32), alpha=1.0, early_stopping=True, max_iter=80, random_state=0)),
}
def ts(c): s = c.sum(0); return s[0]/max(s[0]+s[1],1) - s[2]/max(s[2]+s[3],1)

rng = np.random.RandomState(0); out = []; t0 = time.time()
folds = {"random": list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y)),
         "grouped": list(GroupKFold(5).split(X, y, codes))}
for mn, mkm in TUNED.items():
    Cs = {}
    for sn, fl in folds.items():
        C = np.zeros((NA, 4))
        for a, te in fl:
            if sn == "random": v = rng.rand(len(a)) < .2
            else:
                ua = np.unique(codes[a]); v = np.isin(codes[a], rng.choice(ua, int(.2*len(ua)), replace=False))
            m = mkm().fit(X[a[~v]], y[a[~v]]); t = best_t(m.predict_proba(X[a[v]])[:, 1], y[a[v]])
            add_counts(C, te, m.predict_proba(X[te])[:, 1], t)
        Cs[sn] = C
    keep = (Cs["random"].sum(1) > 0) & (Cs["grouped"].sum(1) > 0); R_, G_ = Cs["random"][keep], Cs["grouped"][keep]
    idx = np.random.RandomState(1).randint(0, len(R_), (300, len(R_)))
    d = np.array([ts(R_[i]) - ts(G_[i]) for i in idx]); br = np.array([ts(R_[i]) for i in idx]); bg = np.array([ts(G_[i]) for i in idx])
    out.append(dict(model=mn, TSS_random=round(ts(R_), 3), CI_r=f"[{np.percentile(br,2.5):.2f},{np.percentile(br,97.5):.2f}]",
                    TSS_grouped=round(ts(G_), 3), CI_g=f"[{np.percentile(bg,2.5):.2f},{np.percentile(bg,97.5):.2f}]",
                    inflation=round(ts(R_) - ts(G_), 3), inflation_CI=f"[{np.percentile(d,2.5):+.3f},{np.percentile(d,97.5):+.3f}]"))
    print(mn, "done", round(time.time() - t0), "s", flush=True)
R = pd.DataFrame(out); print(R.to_string(index=False))
R.to_csv("leakage_tuned_5fold.csv", index=False)

### 3.3 Cross-cycle transfer, default hyperparameters (saves cycle_transfer_results.csv)

In [ ]:
from sklearn.metrics import roc_auc_score
A = pd.read_pickle("c24p_samples.pkl"); B = pd.read_pickle("c25_samples.pkl")
FE = [x for x in A.columns if x.endswith(("_last", "_mean", "_std", "_delta"))]

a_tr0 = A[A.time < "2014-01-01"]; a_te = A[(A.time >= "2014-01-15") & (~A.ar.isin(set(a_tr0.ar)))]
b_tr0 = B[B.time < "2023-01-01"]; b_te = B[(B.time >= "2023-01-15") & (~B.ar.isin(set(b_tr0.ar)))]
def val_split(D, seed=0):
    ars = D.ar.unique(); v = set(np.random.RandomState(seed).choice(ars, int(.2*len(ars)), replace=False))
    return D[~D.ar.isin(v)], D[D.ar.isin(v)]
a_tr, a_va = val_split(a_tr0); b_tr, b_va = val_split(b_tr0)
print("C24p train/val/test:", len(a_tr), len(a_va), len(a_te), "pos rate test", round(a_te.y.mean(), 4), "ARs", a_te.ar.nunique())
print("C25  train/val/test:", len(b_tr), len(b_va), len(b_te), "pos rate test", round(b_te.y.mean(), 4), "ARs", b_te.ar.nunique())

def mk2(n):
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=300)) if n == "LR" \
        else HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, class_weight="balanced", random_state=0)
def ar_counts(p, yy, ar, t):
    codes_, _ = pd.factorize(ar); n = codes_.max() + 1; pr = p >= t
    f = lambda w: np.bincount(codes_, weights=w.astype(float), minlength=n)
    return np.c_[f(pr&(yy==1)), f((~pr)&(yy==1)), f(pr&(yy==0)), f((~pr)&(yy==0))]
def tss_c(c): s = c.sum(0); return s[0]/max(s[0]+s[1],1) - s[2]/max(s[2]+s[3],1)

EX = [("A in-cycle C24 (2010-13 -> 2014-15)", a_tr, a_va, a_te),
      ("B cross  C24 -> C25 late",            a_tr, a_va, b_te),
      ("C in-cycle C25 (2021-22 -> 2023-24)", b_tr, b_va, b_te),
      ("D reverse C25 -> C24 late",           b_tr, b_va, a_te)]
rng = np.random.RandomState(1); NB = 300
rows, cnts = [], {}
for mn in ["LR", "HGB"]:
    for nm, tr, va, te in EX:
        m = mk2(mn).fit(tr[FE].values, tr.y.values)
        t = best_t(m.predict_proba(va[FE].values)[:, 1], va.y.values)
        p = m.predict_proba(te[FE].values)[:, 1]; yy = te.y.values
        c = ar_counts(p, yy, te.ar.values, t); cnts[(mn, nm[0])] = c
        idx = rng.randint(0, len(c), (NB, len(c))); bs = np.array([tss_c(c[i]) for i in idx])
        rows.append(dict(model=mn, setup=nm, n_test=len(te), pos_rate=round(yy.mean(), 3), TSS=round(tss_c(c), 3),
                         CI95=f"[{np.percentile(bs,2.5):.3f}, {np.percentile(bs,97.5):.3f}]", AUC=round(roc_auc_score(yy, p), 3)))
print(pd.DataFrame(rows).to_string(index=False))
for mn in ["LR", "HGB"]:
    cb, cc = cnts[(mn, "B")], cnts[(mn, "C")]
    idx = np.random.RandomState(2).randint(0, len(cb), (NB, len(cb)))
    d = np.array([tss_c(cb[i]) - tss_c(cc[i]) for i in idx])
    print(f"{mn}: TSS(cross C24->C25) - TSS(in-cycle C25) = {tss_c(cb)-tss_c(cc):+.3f}  CI95 [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")
pd.DataFrame(rows).to_csv("cycle_transfer_results.csv", index=False)

### 3.4 Hyperparameter tuning (grouped 3-fold CV on 2010-2013) and tuned results

In [ ]:
import itertools
def mk3(name, **kw):
    if name == "LR": return make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=400, **kw))
    if name == "HGB": return HistGradientBoostingClassifier(class_weight="balanced", random_state=0, learning_rate=0.06, **kw)
    return make_pipeline(StandardScaler(), MLPClassifier(early_stopping=True, max_iter=80, random_state=0, **kw))
GRID = {
 "LR":  [dict(C=c) for c in (0.003, 0.03, 0.3)],
 "HGB": [dict(max_iter=i, max_leaf_nodes=l, min_samples_leaf=s, l2_regularization=r) for i, l, s, r in
         [(100, 8, 200, 1), (300, 8, 200, 10), (100, 31, 500, 10), (300, 31, 500, 10), (150, 15, 100, 1), (150, 15, 500, 10), (300, 8, 500, 1), (100, 31, 50, 1)]],
 "MLP": [dict(hidden_layer_sizes=h, alpha=a) for h, a in [((32,), 1e-3), ((32,), 1e-1), ((32,), 1.0), ((32,), 10.0), ((64, 32), 1.0)]],
}
tr_all = A[A.time < "2014-01-01"].reset_index(drop=True)
tune = tr_all.iloc[::3]
g = pd.factorize(tune.ar)[0]; Xt, yt = tune[FE].values, tune.y.values
best = {}; t0 = time.time()
for mn, grid in GRID.items():
    sc = []
    for kw in grid:
        aucs = [roc_auc_score(yt[v], mk3(mn, **kw).fit(Xt[a], yt[a]).predict_proba(Xt[v])[:, 1]) for a, v in GroupKFold(3).split(Xt, yt, g)]
        sc.append(np.mean(aucs))
    best[mn] = grid[int(np.argmax(sc))]
    print(mn, "best:", best[mn], "| grouped-CV AUC", [round(s, 3) for s in sc], round(time.time() - t0), "s", flush=True)

def ci(c, nb=200):
    idx = np.random.RandomState(1).randint(0, len(c), (nb, len(c))); b = [tss_c(c[i]) for i in idx]; return f"[{np.percentile(b,2.5):.2f},{np.percentile(b,97.5):.2f}]"
a_tr, a_va = val_split(tr_all)
a_te = A[(A.time >= "2014-01-15") & (~A.ar.isin(set(tr_all.ar)))]
b_tr0 = B[B.time < "2023-01-01"]; b_te = B[(B.time >= "2023-01-15") & (~B.ar.isin(set(b_tr0.ar)))]
rows = []
for mn in GRID:
    m = mk3(mn, **best[mn]).fit(a_tr[FE].values, a_tr.y.values)
    t = best_t(m.predict_proba(a_va[FE].values)[:, 1], a_va.y.values)
    for nm, D in [("C24 forward (2014-15)", a_te), ("C25 late (2023-24)", b_te)]:
        p = m.predict_proba(D[FE].values)[:, 1]; c = ar_counts(p, D.y.values, D.ar.values, t)
        rows.append(dict(model=mn, test=nm, TSS=round(tss_c(c), 3), CI95=ci(c), AUC=round(roc_auc_score(D.y, p), 3)))
    rr = np.random.RandomState(5).rand(len(A)); te_r = A[rr < .2]; tr_r = A[rr >= .2]
    trr, var = tr_r[np.random.RandomState(6).rand(len(tr_r)) >= .2], tr_r[np.random.RandomState(6).rand(len(tr_r)) < .2]
    ua = A.ar.unique(); tes = set(np.random.RandomState(7).choice(ua, int(.2*len(ua)), replace=False))
    te_g = A[A.ar.isin(tes)]; tr_g, va_g = val_split(A[~A.ar.isin(tes)], 3)
    for nm, (tr, va, te) in [("random 80/20 rows (leaky)", (trr, var, te_r)), ("AR-grouped 80/20", (tr_g, va_g, te_g))]:
        mm = mk3(mn, **best[mn]).fit(tr[FE].values, tr.y.values); tt = best_t(mm.predict_proba(va[FE].values)[:, 1], va.y.values)
        p = mm.predict_proba(te[FE].values)[:, 1]; c = ar_counts(p, te.y.values, te.ar.values, tt)
        rows.append(dict(model=mn, test=nm, TSS=round(tss_c(c), 3), CI95=ci(c), AUC=round(roc_auc_score(te.y, p), 3)))
R = pd.DataFrame(rows); print(R.to_string(index=False))
R.to_csv("tuned_models_results.csv", index=False)

### 3.5 Per-year skill in Cycle 25, calibration and recalibration (models trained on Cycle 24 only)
Copy the printed per-year and calibration numbers into the figure cell (section 4) if they change.

In [ ]:
from sklearn.metrics import brier_score_loss
A = pd.read_pickle("c24p_samples.pkl"); B = pd.read_pickle("c25_samples.pkl")
a_tr0 = A[A.time < "2014-01-01"]; a_tr, a_va = val_split(a_tr0)
b_tr0 = B[B.time < "2023-01-01"]; _, b_va = val_split(b_tr0)
b_te = B[(B.time >= "2023-01-15") & (~B.ar.isin(set(b_tr0.ar)))]

lg = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6) / (1 - np.clip(p, 1e-6, 1-1e-6))).reshape(-1, 1)
def ece(p, yy, k=10):
    q = pd.qcut(p, k, duplicates="drop"); d = pd.DataFrame({"p": p, "y": yy, "q": q}).groupby("q", observed=True).agg(p=("p","mean"), y=("y","mean"), n=("y","size"))
    return float((d.n * (d.p - d.y).abs()).sum() / d.n.sum())

clim = a_tr.y.mean()
for mn in ["LR", "HGB"]:
    m = mk2(mn).fit(a_tr[FE].values, a_tr.y.values)
    pv = m.predict_proba(a_va[FE].values)[:, 1]; thr = best_t(pv, a_va.y.values)
    print(f"\n===== {mn}: trained on Cycle 24 (2010-13), threshold from C24 validation =====")
    rows = []
    for yr in [2021, 2022, 2023, 2024]:
        D = B[B.time.dt.year == yr]; p = m.predict_proba(D[FE].values)[:, 1]; yy = D.y.values
        c = ar_counts(p, yy, D.ar.values, thr)
        rows.append(dict(year=yr, n=len(D), pos_rate=round(yy.mean(), 3), TSS_fixed=round(tss_c(c), 3), CI95=ci(c),
                         TSS_oracle=round(tss_pt(p, yy, best_t(p, yy)), 3), AUC=round(roc_auc_score(yy, p), 3)))
    print(pd.DataFrame(rows).to_string(index=False))
    pt = m.predict_proba(b_te[FE].values)[:, 1]; yt = b_te.y.values
    pl24 = LogisticRegression(C=1e6).fit(lg(pv), a_va.y.values)
    pb = m.predict_proba(b_va[FE].values)[:, 1]; pl25 = LogisticRegression(C=1e6).fit(lg(pb), b_va.y.values)
    ref = brier_score_loss(yt, np.full(len(yt), clim))
    cal = []
    for nm, pp in [("raw", pt), ("Platt on C24", pl24.predict_proba(lg(pt))[:, 1]), ("recalibrated on C25 early", pl25.predict_proba(lg(pt))[:, 1])]:
        b = brier_score_loss(yt, pp); cal.append(dict(version=nm, Brier=round(b, 4), BSS_vs_C24_clim=round(1 - b/ref, 3), ECE=round(ece(pp, yt), 3), mean_pred=round(pp.mean(), 3), obs_rate=round(yt.mean(), 3)))
    print("\nCalibration on C25 late (2023-24):"); print(pd.DataFrame(cal).to_string(index=False))

## 4. Figures
Figures 1-2 read the CSV files written above. Figures 3-4 use the printed numbers of section 3.5 (typed in below).

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("figs", exist_ok=True)
BLUE, ORANGE, INK, INK2, GRID_ = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False})
def pci(s): lo, hi = [float(x) for x in str(s).strip("[]").split(",")]; return lo, hi
def style(ax): ax.grid(axis="y", color=GRID_, lw=0.8); ax.set_axisbelow(True)
def save(fig, name): fig.savefig(f"figs/{name}.png", dpi=300, bbox_inches="tight", facecolor="white"); plt.show()

U = pd.read_csv("leakage_audit_c24p.csv"); T = pd.read_csv("leakage_tuned_5fold.csv")
def panel(ax, labels, rv, rci, gv, gci, title):
    x = np.arange(len(labels)); w = 0.36
    for off, v, c, colr, nm in [(-w/2 - 0.01, rv, rci, BLUE, "Random row split (leaky)"), (w/2 + 0.01, gv, gci, ORANGE, "Grouped by active region")]:
        lo = [a - pci(b)[0] for a, b in zip(v, c)]; hi = [pci(b)[1] - a for a, b in zip(v, c)]
        ax.bar(x + off, v, w, color=colr, edgecolor="white", linewidth=2, label=nm, yerr=[lo, hi], error_kw=dict(ecolor=INK2, lw=1, capsize=2))
        for xi, vi, h in zip(x + off, v, hi): ax.text(xi, vi + h + 0.012, f"{vi:.2f}", ha="center", fontsize=8, color=INK)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.5); ax.set_ylim(0, 1.08); ax.set_ylabel("TSS (24 h, M+)"); ax.set_title(title, fontsize=10, loc="left"); style(ax)
fig, axs = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [5, 3]})
mods = ["1-feat (USFLUX)", "3-feat (USFLUX,TOTUSJH,R_VALUE)", "LR", "HGB", "MLP"]
r_ = U[U.split.str.startswith("random")].set_index("model").loc[mods]; g_ = U[U.split.str.startswith("grouped")].set_index("model").loc[mods]
panel(axs[0], ["1-feature", "3-feature", "LR", "HGB", "MLP"], r_.TSS.values, r_.CI95.values, g_.TSS.values, g_.CI95.values, "(a) Default hyperparameters (not tuned)")
tm = ["LR (tuned)", "HGB (tuned)", "MLP (tuned)"]; T = T.set_index("model").loc[tm]
panel(axs[1], ["LR", "HGB", "MLP"], T.TSS_random.values, T.CI_r.values, T.TSS_grouped.values, T.CI_g.values, "(b) Tuned with grouped CV")
axs[0].legend(frameon=False, fontsize=8.5, loc="upper left", ncol=2); fig.suptitle("Leakage inflates TSS, mostly for under-regularized flexible models (Cycle 24, 5-fold, 95% CI over active regions)", fontsize=10, x=0.01, ha="left")
save(fig, "fig1_leakage")

Xc = pd.read_csv("cycle_transfer_results.csv"); Xc["k"] = Xc.setup.str[0]
labs = {"A": "In-cycle C24\n(2010-13 -> 2014-15)", "B": "Cross-cycle\nC24 -> C25 late", "C": "In-cycle C25\n(2021-22 -> 2023-24)", "D": "Reverse\nC25 -> C24 late"}
fig, ax = plt.subplots(figsize=(8.5, 4))
for off, mn, colr in [(-0.08, "LR", BLUE), (0.08, "HGB", ORANGE)]:
    d = Xc[Xc.model == mn].set_index("k").loc[list(labs)]
    lo = [t - pci(c)[0] for t, c in zip(d.TSS, d.CI95)]; hi = [pci(c)[1] - t for t, c in zip(d.TSS, d.CI95)]
    ax.errorbar(np.arange(4) + off, d.TSS, yerr=[lo, hi], fmt="o", color=colr, ecolor=colr, elinewidth=1.4, capsize=3, ms=7, mec="white", mew=1.5, label=mn)
ax.set_xticks(range(4)); ax.set_xticklabels(list(labs.values()), fontsize=8.5); ax.set_ylim(0.4, 0.85); ax.set_ylabel("TSS (24 h, M+)"); style(ax); ax.legend(frameon=False)
ax.set_title("Training on Cycle 24: no measurable TSS loss on Cycle 25\n(paired dTSS cross - in-cycle: LR +0.011 [-0.009, +0.031]; HGB +0.009 [-0.013, +0.031])", fontsize=9, loc="left")
save(fig, "fig2_cross_cycle")

# values below are copied from the section 3.5 output
yrs = [2021, 2022, 2023, 2024]
D3 = {"LR": ([0.605, 0.680, 0.633, 0.663], [(0.43, 0.77), (0.62, 0.74), (0.58, 0.68), (0.62, 0.70)], BLUE),
      "HGB": ([0.598, 0.635, 0.565, 0.623], [(0.44, 0.76), (0.58, 0.69), (0.50, 0.62), (0.58, 0.66)], ORANGE)}
fig, ax = plt.subplots(figsize=(7, 4))
for off, (mn, (v, c, colr)) in zip([-0.04, 0.04], D3.items()):
    ax.errorbar(np.array(yrs) + off, v, yerr=[[a - b[0] for a, b in zip(v, c)], [b[1] - a for a, b in zip(v, c)]], fmt="-o", color=colr, lw=1.6, capsize=3, ms=6, mec="white", mew=1.2, label=mn)
ax.set_xticks(yrs); ax.set_ylim(0.3, 0.85); ax.set_ylabel("TSS (fixed C24 threshold)"); style(ax); ax.legend(frameon=False)
ax.set_title("No downward trend in Cycle 25 (models trained on Cycle 24 only)", fontsize=10, loc="left")
save(fig, "fig3_per_year")

vers = ["Raw\n(balanced weights)", "Platt fit\non Cycle 24", "Recalibrated on\nCycle 25 early"]
bss = {"LR": [-0.442, 0.349, 0.339], "HGB": [0.198, 0.251, 0.288]}
fig, ax = plt.subplots(figsize=(7.5, 4)); x = np.arange(3); w = 0.36
for off, (mn, colr) in zip([-w/2 - 0.01, w/2 + 0.01], [("LR", BLUE), ("HGB", ORANGE)]):
    ax.bar(x + off, bss[mn], w, color=colr, edgecolor="white", linewidth=2, label=mn)
    for xi, vi in zip(x + off, bss[mn]): ax.text(xi, vi + (0.015 if vi >= 0 else -0.03), f"{vi:.2f}", ha="center", fontsize=8.5)
ax.axhline(0, color=INK2, lw=1); ax.set_ylim(-0.52, 0.42); ax.set_xticks(x); ax.set_xticklabels(vers, fontsize=8.5); ax.set_ylabel("Brier skill score (C25 2023-24)"); style(ax); ax.legend(frameon=False)
ax.set_title("Recalibration raises Brier skill under the Cycle 25 base-rate shift", fontsize=10, loc="left")
save(fig, "fig4_calibration")

In [ ]:
# final checkpoint of results and figures to Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for p in ["leakage_audit_c24p.csv", "leakage_tuned_5fold.csv", "cycle_transfer_results.csv", "tuned_models_results.csv"]:
        if os.path.exists(p): shutil.copy(p, f"{BACKUP}/{p}")
    os.makedirs(f"{BACKUP}/figs", exist_ok=True)
    for f in os.listdir("figs"): shutil.copy(f"figs/{f}", f"{BACKUP}/figs/{f}")
    print("results and figures saved to", BACKUP)
except Exception as e: print("save skipped:", type(e).__name__)

## Notes
- All numbers in the paper come from the cells above. Seeds are fixed (random states 0-7).
- The tuned-model leakage test (3.2) uses the parameters selected in 3.4; if the grid or data change, update the `TUNED` dictionary.
- Limitations are listed in the paper (Section 5): Cycle 25 is incomplete, 17 SHARP parameters only, one target (M+, 24 h), two flare catalogues, overlapping windows.